# Chapter 5 — 프로토콜 취약점과 트래픽 분석 (실습)

## 실습 목표
1. 합성 SYN Flood 트래픽에서 SYN/ACK 비율 비정상을 탐지한다.
2. ARP 패킷에서 같은 IP 에 다중 MAC 응답을 검출한다.
3. DNS 트래픽에서 응답 수가 쿼리 수보다 많은 도메인을 찾는다.
4. 위 세 탐지 함수를 작은 라이브러리로 패키징한다.



In [ ]:
# 파트별 노트북은 같은 종류의 공통 데이터·모델 작업 폴더를 사용합니다.
from pathlib import Path as _BookPath
import os as _book_os
_book_cwd = _BookPath.cwd()
if (_book_cwd.name in {"part1", "part2", "part3", "part4"}
        and _book_cwd.parent.name in {"instructor", "exercises", "solutions"}):
    _book_os.chdir(_book_cwd.parent)


In [ ]:
# 첫 사용시에만 주석 해제하여 실행 (Colab/venv 환경에 따라 일부는 이미 설치됨)
# !pip install -q scapy cryptography   # cryptography: scapy TLS 레이어용

In [ ]:
# === 한글 폰트 설정 (matplotlib) ===
from pathlib import Path
import matplotlib
from matplotlib import font_manager
for fp in ['/usr/share/fonts/truetype/nanum/NanumGothic.ttf',
           '/usr/share/fonts/truetype/nanum/NanumBarunGothic.ttf']:
    if Path(fp).exists():
        font_manager.fontManager.addfont(fp)
        matplotlib.rcParams['font.family'] = font_manager.FontProperties(fname=fp).get_name()
        break
matplotlib.rcParams['axes.unicode_minus'] = False

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from collections import defaultdict
from pathlib import Path
from scapy.all import (
    Ether, IP, TCP, UDP, ARP, DNS, DNSQR, DNSRR, wrpcap, rdpcap, Raw
)

---
## Step 1. 합성 SYN Flood PCAP 만들기 + 탐지

In [ ]:
# ── 실습 과제 ──
# 목표: Step 1. 합성 SYN Flood PCAP 만들기 + 탐지
# 만들 결과: rng, victim_ip, packets, src, sport, OUT
# 산출: 결과 값 출력
# 아래에 직접 구현하세요. 막히면 다음 셀의 '힌트 보기'(구조 스켈레톤)를 참고합니다.

<details><summary>💡 <b>힌트 보기</b> (막혔을 때만)</summary>

> 정답이 아닌 **구조 스켈레톤**입니다. 변수명·제어흐름·주석만 노출되고 실제 값/로직은 `...` 로 가려둠.

```python
# 정상 트래픽 200개 + SYN Flood 800개 혼합
rng = ...
victim_ip = ...
packets = ...

# 정상 완전 3-way handshake 100세트 (300 packets)
for _ in range(100):
    src = ...
    sport = ...
    packets.append(IP(src = ...
    packets.append(IP(src = ...
    packets.append(IP(src = ...

# SYN Flood (응답 없는 SYN) 700개
for _ in range(700):
    src = ...
    sport = ...
    packets.append(IP(src = ...

...
OUT = ...
...
```
</details>


In [ ]:
# ── 실습 과제 ──
# 목표: Step 1. 합성 SYN Flood PCAP 만들기 + 탐지
# 만들 결과: detect_syn_flood(), pkts, flags, ratio, result
# 산출: 결과 값 출력
# 아래에 직접 구현하세요. 막히면 다음 셀의 '힌트 보기'(구조 스켈레톤)를 참고합니다.

<details><summary>💡 <b>힌트 보기</b> (막혔을 때만)</summary>

> 정답이 아닌 **구조 스켈레톤**입니다. 변수명·제어흐름·주석만 노출되고 실제 값/로직은 `...` 로 가려둠.

```python
def detect_syn_flood(pcap_path: ...
    ...
    pkts = ...
    n_syn, n_synack = ...
    for p in pkts:
        if IP not in p or TCP not in p:
            continue
        flags = ...
        if p[IP].dst == dst_ip and 'S' in flags and 'A' not in flags:
            n_syn += ...
        if p[IP].src == dst_ip and 'S' in flags and 'A' in flags:
            n_synack += ...
    ratio = ...
    return ...

result = ...
...
```
</details>


In [ ]:
# SYN Flood 시각화 — 정상은 SYN ≈ SYN-ACK, 플러드는 SYN 폭증
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(5, 3.3))
ax.bar(['SYN', 'SYN-ACK'], [result['syn'], result['syn_ack']], color=['#e74c3c', '#1a5f7a'])
ax.set_title(f"SYN Flood 탐지 — 비율 {result['ratio']:.1f} ({'의심' if result['flood'] else '정상'})")
ax.set_ylabel('패킷 수')
plt.tight_layout(); plt.show()

관찰: 정상은 1:1 비율이어야 하지만, SYN Flood 포함된 PCAP 는 SYN/SYN+ACK 비율이 크게 치우친다.

---
## Step 2. 합성 ARP Spoof + 탐지

In [ ]:
# ── 실습 과제 ──
# 목표: Step 2. 합성 ARP Spoof + 탐지
# 만들 결과: arp_pkts, ARP_OUT, detect_arp_spoof(), ip_to_macs, suspects
# 산출: 결과 값 출력
# 아래에 직접 구현하세요. 막히면 다음 셀의 '힌트 보기'(구조 스켈레톤)를 참고합니다.

<details><summary>💡 <b>힌트 보기</b> (막혔을 때만)</summary>

> 정답이 아닌 **구조 스켈레톤**입니다. 변수명·제어흐름·주석만 노출되고 실제 값/로직은 `...` 로 가려둠.

```python
arp_pkts = ...
# 정상 ARP 응답: 10.0.0.1 = AA:AA:AA:AA:AA:AA (Gateway)
for _ in range(20):
    ...
# 공격자 위조 ARP: 10.0.0.1 = EE:EE:EE:EE:EE:EE
for _ in range(15):
    ...
# 정상 ARP 응답: 10.0.0.50 = BB:BB:BB:BB:BB:BB
for _ in range(5):
    ...

...
ARP_OUT = ...
...

def detect_arp_spoof(pcap_path: ...
    ...
    ip_to_macs = ...
    for p in rdpcap(pcap_path):
        if ARP in p and p[ARP].op == 2: ...
            ...
    return ...

suspects = ...
...
for ip, macs in suspects.items():
    ...
```
</details>


---
## Step 3. 합성 DNS Poison + 탐지

In [ ]:
# ── 실습 과제 ──
# 목표: Step 3. 합성 DNS Poison + 탐지
# 만들 결과: dns_pkts, tid, an, DNS_OUT, detect_dns_poison(), queries
# 산출: 결과 값 출력
# 아래에 직접 구현하세요. 막히면 다음 셀의 '힌트 보기'(구조 스켈레톤)를 참고합니다.

<details><summary>💡 <b>힌트 보기</b> (막혔을 때만)</summary>

> 정답이 아닌 **구조 스켈레톤**입니다. 변수명·제어흐름·주석만 노출되고 실제 값/로직은 `...` 로 가려둠.

```python
dns_pkts = ...
# 정상: 1 쿼리 -> 1 응답
for i, dom in enumerate(['example.com', 'google.com', 'github.com'] * 5):
    tid = ...
    ...
                                            an = ...
# 공격자: bank.com 1 쿼리에 3개 위조 응답 (서로 다른 IP)
...
for fake_ip in ['6.6.6.6', '7.7.7.7', '8.8.8.8']:
    ...
                                            an = ...

...
DNS_OUT = ...
...

def detect_dns_poison(pcap_path: ...
    ...
    queries = ...
    responses = ...
    for p in rdpcap(pcap_path):
        if DNS not in p:
            continue
        d = ...
        if d.qd is None:
            continue
        qname = ...
        if d.qr == 0:
            queries[qname] += ...
        else:
            responses[qname] += ...
    suspects = ...
                for dom in responses if responses[dom] > queries.get(dom, 0)}
    return ...

suspects = ...
...
for dom, (n_resp, n_query) in suspects.items():
    ...
```
</details>


---
## Step 4. TLS ClientHello 와 JA3 핑거프린트

본문 §5.4.3 의 JA3 를 직접 계산한다. JA3 는 TLS 핸드셰이크의 **첫 메시지(ClientHello)** 에 담긴
TLS 버전·암호군·확장·타원곡선·포인트 형식을 순서대로 이어 붙인 뒤 MD5 로 해시한 값이다.
암호화가 시작되기 **전에** 평문으로 오가는 메시지라 **복호화 없이** 계산할 수 있다.

같은 사이트(`www.example.com`)에 접속하는 두 클라이언트를 만들어 JA3 를 비교한다.

In [ ]:
from scapy.all import load_layer
load_layer('tls')
from scapy.layers.tls.handshake import TLSClientHello
from scapy.layers.tls.extensions import (TLS_Ext_ServerName, ServerName,
                                         TLS_Ext_SupportedGroups, TLS_Ext_SupportedPointFormat)
import hashlib

# GREASE(0x0a0a 등)는 브라우저가 무작위로 끼워 넣는 값이라 JA3 계산에서 제외한다
GREASE = {0x0a0a, 0x1a1a, 0x2a2a, 0x3a3a, 0x4a4a, 0x5a5a, 0x6a6a, 0x7a7a,
          0x8a8a, 0x9a9a, 0xaaaa, 0xbaba, 0xcaca, 0xdada, 0xeaea, 0xfafa}

def build_client_hello(version, ciphers, groups, point_formats, sni):
    exts = [TLS_Ext_ServerName(servernames=[ServerName(servername=sni)]),
            TLS_Ext_SupportedGroups(groups=groups),
            TLS_Ext_SupportedPointFormat(ecpl=point_formats)]
    return TLSClientHello(version=version, ciphers=ciphers, ext=exts)

# ── 실습 과제 ──
# 목표: Step 4. TLS ClientHello 와 JA3 핑거프린트 — 본문 §5.4.3 의 JA3 를 직접 계산한다. JA3 는 TLS 핸드셰이크의 **첫 메시지(ClientHello)** 에 담긴
# 만들 결과: ja3_fingerprint(), h, ciphers, exts, groups, point_formats
# 산출: 결과 값 출력
# 아래에 직접 구현하세요. 막히면 다음 셀의 '힌트 보기'(구조 스켈레톤)를 참고합니다.

# 최신 브라우저: TLS 1.3 AEAD 암호군 + ECDHE-GCM, x25519 우선
browser = build_client_hello(0x0303,
    [0x1301, 0x1302, 0x1303, 0xc02b, 0xc02f, 0xc02c, 0xc030],
    [0x001d, 0x0017, 0x0018], [0], 'www.example.com')
# 오래된 라이브러리 기반 자동화 도구: 구형 RSA·CBC 암호군만 제시
tool = build_client_hello(0x0303,
    [0x0035, 0x002f, 0x000a],
    [0x0017], [0], 'www.example.com')

ja3_table = []
for name, hello in [('browser', browser), ('tool', tool)]:
    ja3_str, digest = ja3_fingerprint(bytes(hello))
    ja3_table.append({'client': name, 'sni': 'www.example.com', 'ja3': digest, 'ja3_string': ja3_str})
    print(f'{name:<8} JA3 = {digest}')
    print(f'{"":<8} 문자열 = {ja3_str}')

pd.DataFrame(ja3_table)[['client', 'sni', 'ja3']]

<details><summary>💡 <b>힌트 보기</b> (막혔을 때만)</summary>

> 정답이 아닌 **구조 스켈레톤**입니다. 변수명·제어흐름·주석만 노출되고 실제 값/로직은 `...` 로 가려둠.

```python
def ja3_fingerprint(hello_bytes: ...
    ...
    h = ...
    ciphers = ...
    exts = ...
    groups, point_formats = ...
    for e in (h.ext or []):
        if isinstance(e, TLS_Ext_SupportedGroups):
            groups = ...
        if isinstance(e, TLS_Ext_SupportedPointFormat):
            point_formats = ...
    ja3_str = ...
                        ...
    return ...
```
</details>


---
## Step 5. 작은 탐지 도구 패키징

In [ ]:
# ── 실습 과제 ──
# 목표: Step 5. 작은 탐지 도구 패키징
# 만들 결과: analyze_pcap()
# 산출: 결과 값 출력
# 아래에 직접 구현하세요. 막히면 다음 셀의 '힌트 보기'(구조 스켈레톤)를 참고합니다.

<details><summary>💡 <b>힌트 보기</b> (막혔을 때만)</summary>

> 정답이 아닌 **구조 스켈레톤**입니다. 변수명·제어흐름·주석만 노출되고 실제 값/로직은 `...` 로 가려둠.

```python
def analyze_pcap(pcap_path: ...
    return ...
        ...

# 세 PCAP 모두 분석
for path in [str(Path.cwd() / n) for n in ['syn_flood.pcap', 'arp_spoof.pcap', 'dns_poison.pcap']]:
    ...
    for k, v in analyze_pcap(path).items():
        ...
```
</details>


---
## 정리

- SYN/SYN+ACK 비율, IP-MAC 매핑, DNS 응답/쿼리 비율 — 단순한 통계지표만으로도 1차 탐지가 가능하다.
- TLS 는 복호화하지 않아도 ClientHello 로 클라이언트를 식별할 수 있다(JA3). 목적지가 같아도 소프트웨어가 다르면 해시가 달라진다.
- 이런 룰 기반 탐지는 빠르고 해석 가능하지만, 임계값 우회·노이즈에 취약하다.
- 다음 챕터에서는 PCAP·NetFlow·Zeek 의 형식을 비교하고, 더 풍부한 feature 를 추출하는 파이프라인을 만든다.